# Experiment 4: Generative Adversarial Network (GAN)

**Name:** Noora  
**Roll No:** 2301420015  
**Course:** B.Tech CSE (Data Science)

---

## Objective
To implement a basic GAN that generates handwritten digit images similar to MNIST.

## Theory
A **GAN** consists of two competing neural networks:
- **Generator (G)**: Takes random noise z ~ N(0,1) as input and generates fake images
- **Discriminator (D)**: Classifies images as real (1) or fake (0)

They are trained adversarially:
- D tries to correctly distinguish real vs fake
- G tries to fool D into classifying its outputs as real

The minimax objective:
```
min_G max_D [E[log D(x)] + E[log(1 - D(G(z)))]]
```

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import matplotlib.pyplot as plt

tf.random.set_seed(42)
np.random.seed(42)

# --- Load Data ---
(x_train, _), (_, _) = keras.datasets.mnist.load_data()
x_train = (x_train.astype('float32') / 127.5) - 1.0  # Normalize to [-1, 1]
x_train = x_train.reshape(-1, 28*28)

LATENT_DIM  = 100
BATCH_SIZE  = 64
EPOCHS      = 200

print(f"Training data shape: {x_train.shape}")

# --- Generator ---
generator = keras.Sequential([
    keras.layers.Dense(256, activation='leaky_relu', input_shape=(LATENT_DIM,)),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(512, activation='leaky_relu'),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(784, activation='tanh'),
    keras.layers.Reshape((28, 28))
], name='Generator')

# --- Discriminator ---
discriminator = keras.Sequential([
    keras.layers.Flatten(input_shape=(28, 28)),
    keras.layers.Dense(512, activation='leaky_relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(256, activation='leaky_relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(1, activation='sigmoid')
], name='Discriminator')

discriminator.compile(
    optimizer=keras.optimizers.Adam(0.0002, beta_1=0.5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

generator.summary()
discriminator.summary()

In [ ]:
# --- GAN (combined) ---
discriminator.trainable = False
gan_input  = keras.Input(shape=(LATENT_DIM,))
gan_output = discriminator(generator(gan_input))
gan = keras.Model(gan_input, gan_output, name='GAN')
gan.compile(
    optimizer=keras.optimizers.Adam(0.0002, beta_1=0.5),
    loss='binary_crossentropy'
)

# --- Training Loop ---
g_losses, d_losses = [], []

for epoch in range(EPOCHS):
    # Sample real images
    idx   = np.random.randint(0, x_train.shape[0], BATCH_SIZE)
    real  = x_train[idx].reshape(BATCH_SIZE, 28, 28)
    # Generate fake images
    noise = np.random.normal(0, 1, (BATCH_SIZE, LATENT_DIM))
    fake  = generator.predict(noise, verbose=0)

    # Labels with label smoothing
    real_labels = np.ones((BATCH_SIZE, 1)) * 0.9
    fake_labels = np.zeros((BATCH_SIZE, 1))

    # Train Discriminator
    discriminator.trainable = True
    d_loss_real = discriminator.train_on_batch(real, real_labels)
    d_loss_fake = discriminator.train_on_batch(fake, fake_labels)
    d_loss = 0.5 * (d_loss_real[0] + d_loss_fake[0])

    # Train Generator
    discriminator.trainable = False
    noise   = np.random.normal(0, 1, (BATCH_SIZE, LATENT_DIM))
    g_loss  = gan.train_on_batch(noise, np.ones((BATCH_SIZE, 1)))

    g_losses.append(g_loss)
    d_losses.append(d_loss)

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} | D Loss: {d_loss:.4f} | G Loss: {g_loss:.4f}")

print("Training complete!")

In [ ]:
# --- Plot Loss Curves ---
plt.figure(figsize=(10, 4))
plt.plot(g_losses, label='Generator Loss', color='blue', alpha=0.7)
plt.plot(d_losses, label='Discriminator Loss', color='red', alpha=0.7)
plt.title('GAN Training Loss', fontsize=14, fontweight='bold')
plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.savefig('exp4_loss.png', dpi=150, bbox_inches='tight')
plt.show()

# --- Generate & Display Images ---
noise = np.random.normal(0, 1, (16, LATENT_DIM))
gen_images = generator.predict(noise, verbose=0)

fig, axes = plt.subplots(4, 4, figsize=(8, 8))
fig.suptitle('GAN Generated Images', fontsize=14, fontweight='bold')
for i, ax in enumerate(axes.flatten()):
    ax.imshow(gen_images[i], cmap='gray')
    ax.axis('off')
plt.tight_layout()
plt.savefig('exp4_generated.png', dpi=150, bbox_inches='tight')
plt.show()

## Result
- The Generator progressively learns to produce digit-like images.
- Generator loss decreases while Discriminator loss stabilizes (Nash equilibrium).
- Label smoothing and Dropout help stabilize training.

## Conclusion
GANs are powerful generative models capable of producing realistic images. The adversarial training dynamic creates a minimax game where both networks improve over time. Real-world applications include image synthesis, data augmentation, and style transfer.